# Lab 10 Tasks: Spatial Interpolation

**Total: 100 points.** Complete each task in the cell under it. Grading is by method and reasoning, not by matching every digit. Your numbers can differ slightly from the key if you use a different random seed or a different version of a package, so write down the seed you used.

Tasks 1 to 5 use the **afternoon** Heat Watch window (`af`, 3 to 4 pm), not the evening window from the walkthrough. Tasks 6 and 7 use the census tracts and block groups.

| Task | Points |
|---|---|
| 1. Describe the data | 10 |
| 2. kNN surfaces | 15 |
| 3. Variogram and model | 15 |
| 4. Kriging at five held-out points | 20 |
| 5. Random and blocked cross-validation | 15 |
| 6. Extensive and intensive variables | 15 |
| 7. Choose a method (open-ended) | 10 |
| Optional: write kriging yourself | 0 |

**Remember to submit both the IPYNB and the HTML export.**

## Setup

Run these cells first. They are the same helpers as in the walkthrough.

In [1]:
import sys, subprocess, warnings
from importlib.metadata import version
warnings.filterwarnings("ignore")
try:
    import pyinterpolate as pi
except ImportError:                       # not installed in this kernel yet
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyinterpolate"])
    import pyinterpolate as pi
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| geopandas", gpd.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__, "| pyinterpolate", version("pyinterpolate"))

Python 3.8.16 | geopandas 0.13.2 | numpy 1.24.3 | matplotlib 3.7.1 | pyinterpolate 1.2.1


In [2]:
import shapely
from shapely.geometry import MultiPoint
from sklearn.neighbors import KNeighborsRegressor

PURPLE, MAGENTA, BLUE = "#7C3AED", "#DB2777", "#2563EB"


def blank(ax):
    """Map style: no ticks, no frame, equal aspect."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    for s in ax.spines.values():
        s.set_visible(False)


def load_heat(window):
    """One Heat Watch window ("am", "af" or "pm"), thinned to one point per 250 m cell. Returns x, y (meters) and F."""
    heat = gpd.read_file(f"data/heat/{window}_trav.shp")
    heat = heat[heat["t_f"] > -9000].to_crs(32614)                  # drop removed readings; metric CRS
    xy = np.column_stack([heat.geometry.x, heat.geometry.y])
    cell = (xy // 250).astype(int)
    _, keep = np.unique(cell, axis=0, return_index=True)            # first point in each 250 m cell
    keep = np.sort(keep)
    return xy[keep], heat["t_f"].to_numpy()[keep], len(heat)


def make_grid(xy, n=60):
    """n by n grid of prediction locations over the points, and a flag for the ones inside their convex hull."""
    (x0, y0), (x1, y1) = xy.min(0), xy.max(0)
    gx, gy = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))
    U = np.column_stack([gx.ravel(), gy.ravel()])
    inside = shapely.contains_xy(MultiPoint(xy).convex_hull, U[:, 0], U[:, 1])
    return gx, gy, U, inside


def predict_knn(xy, t, U, k=15, weights="distance"):
    """k nearest neighbors. k = 1 is nearest neighbor; weights "uniform" or "distance" (inverse distance)."""
    return KNeighborsRegressor(n_neighbors=k, weights=weights).fit(xy, t).predict(U)


def fit_variogram(xy, t, model="exponential", step=1000, max_range=15000):
    """Experimental variogram (1 km lags to 15 km) and a fitted model with pyinterpolate."""
    known = np.column_stack([xy, t])
    ex = pi.build_experimental_variogram(known, step_size=step, max_range=max_range)
    return ex, pi.build_theoretical_variogram(ex, models_group=model)


def predict_krige(xy, t, U, model="exponential", fitted=None, k=16):
    """Ordinary kriging. Returns the prediction and the variance error at each row of U."""
    th = fitted if fitted is not None else fit_variogram(xy, t, model)[1]
    out = pi.interpolate_points(th, unknown_locations=U, known_locations=np.column_stack([xy, t]),
                                no_neighbors=k, allow_approximate_solutions=True, progress_bar=False)
    return out[:, 0], out[:, 1]


def show_surface(ax, gx, gy, z, inside, vmin, vmax, title, cmap="YlOrRd"):
    """Draw a gridded surface; grid cells outside the data's convex hull are left blank."""
    zz = np.where(inside, z, np.nan).reshape(gx.shape)
    im = ax.pcolormesh(gx, gy, zz, cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
    ax.set_title(title)
    blank(ax)
    return im


def block_ids(xy, size=5000):
    """Label each point with the 5 km by 5 km square it falls in."""
    return (xy[:, 0] // size).astype(int) * 1000 + (xy[:, 1] // size).astype(int)

In [3]:
from sklearn.model_selection import KFold, LeaveOneGroupOut


def cv_scores(xy, t, method, groups=None, seed=0, **kw):
    """Cross-validated RMSE and MAE. groups=None: random 10-fold. Otherwise: leave one group (block) out."""
    if groups is None:
        splits = list(KFold(10, shuffle=True, random_state=seed).split(xy))
    else:
        splits = list(LeaveOneGroupOut().split(xy, t, groups))
    pred = np.zeros(len(t))
    for train, test in splits:
        if method == "mean":
            pred[test] = t[train].mean()
        elif method == "knn":
            pred[test] = predict_knn(xy[train], t[train], xy[test], kw.get("k", 15))
        elif method == "krige":
            pred[test] = predict_krige(xy[train], t[train], xy[test])[0]
    err = pred - t
    return float(np.sqrt(np.mean(err ** 2))), float(np.mean(np.abs(err)))

In [4]:
def area_weight(source, target, extensive=(), intensive=()):
    """Move values from source polygons to target polygons by the share of area.
    extensive: counts (each source value is split by area share, so totals are kept).
    intensive: rates and medians (area-weighted average of the pieces that fall in each target)."""
    src = source.reset_index(drop=True).copy()
    src["src_id"], src["src_area"] = src.index, src.area
    tgt = target.reset_index(drop=True)[["geometry"]].copy()
    tgt["tgt_id"] = tgt.index
    cols = list(extensive) + list(intensive)
    pieces = gpd.overlay(src[["src_id", "src_area", *cols, "geometry"]], tgt, how="intersection")
    pieces["piece_area"] = pieces.area
    pieces["share"] = pieces["piece_area"] / pieces["src_area"]      # share of the source polygon in this piece
    out = pd.DataFrame(index=tgt.index)
    for c in extensive:
        out[c] = (pieces[c] * pieces["share"]).groupby(pieces["tgt_id"]).sum()
    for c in intensive:
        p = pieces.dropna(subset=[c])
        out[c] = (p[c] * p["piece_area"]).groupby(p["tgt_id"]).sum() / p.groupby("tgt_id")["piece_area"].sum()
    return out.reindex(tgt.index)

In [5]:
from libpysal.weights import KNN
from esda.moran import Moran


def morans_i(xy, t, k=8, seed=0):
    """Moran's I of the values at the points, with k-nearest-neighbor weights (row-standardized)."""
    w = KNN.from_array(xy, k=k)
    w.transform = "r"
    np.random.seed(seed)
    m = Moran(t, w, permutations=999)
    return m.I, m.p_sim

## Task 1 (10 points). Describe the data

Load the afternoon window with `load_heat("af")`.

a. Report the number of usable readings, the number of points after thinning, the CRS used for the thinned points, and the minimum, maximum and mean temperature.

b. Compute Moran's I of the temperatures with `morans_i(xy, t)` (8 nearest neighbors, `seed=0`). Say what the value means and whether it supports using interpolation on these data.

In [6]:
# Task 1 starter

## Task 2 (15 points). kNN surfaces

Using the afternoon points, build a 60 by 60 grid with `make_grid(xy)` and predict distance-weighted kNN surfaces with $k = 3$, $15$ and $50$. Draw the three surfaces side by side with one shared color scale and a colorbar.

a. Report the minimum and maximum prediction over the cells inside the hull for each $k$.

b. In two or three sentences, explain what increasing $k$ does to the surface and why.

In [8]:
# Task 2 starter

## Task 3 (15 points). Variogram and model

a. Build the experimental variogram of the afternoon temperatures (1 km steps, maximum range 15 km) with `pi.build_experimental_variogram`, and plot it with the variance of the data as a dashed line.

b. Fit a **spherical** model with `pi.build_theoretical_variogram(..., models_group="spherical")`. Report the nugget, the (partial) sill, the range in km, and the plateau (nugget plus sill).

c. In two or three sentences, say what the nugget means for these data and what share of the plateau it is.

In [10]:
# Task 3 starter

## Task 4 (20 points). Kriging at five held-out points

Hold out five afternoon points and predict them from the rest.

```python
rng = np.random.default_rng(10)
test = rng.choice(len(t), 5, replace=False)
train = np.setdiff1d(np.arange(len(t)), test)
```

a. Fit an **exponential** model on the training points only, and predict the five held-out points with ordinary kriging (`predict_krige`, passing `fitted=`). Also predict them with distance-weighted kNN, $k = 15$.

b. Make a table with the measured value, the kriging prediction, the kNN prediction, the kriging variance, and the distance in meters from each held-out point to its nearest training point.

c. Which method has the smaller error over these five points? Which point has the highest kriging variance, and what is it about that point that explains it? (Five points are too few to rank the methods in general. Say so.)

In [12]:
# Task 4 starter

## Task 5 (15 points). Random and blocked cross-validation

Using `cv_scores`, compute RMSE (random 10-fold with `seed=0`, and 5 km blocks from `block_ids(xy)`) for the training mean, kNN with $k = 15$, and ordinary kriging, on the afternoon points.

a. Present the six RMSE values in one table (rows: methods; columns: random and blocked).

b. In one paragraph, say which number a city planner should trust for a map of the whole city, and why the two columns differ.

In [14]:
# Task 5 starter

## Task 6 (15 points). Extensive and intensive variables

Load the tracts and block groups as in Part 6 of the walkthrough (`data/dallas_tract_2020.gpkg` and `data/dallas_bg_2020.gpkg`). Use `area_weight` to move the tract population `pop` (extensive) and the tract `median_age` (intensive) onto the block groups.

a. Report the total population before and after, and the minimum and maximum median age before and after. Can any block group have a median age outside the tract range?

b. Explain why population is split and summed while median age is averaged. Then show what goes wrong if median age is wrongly treated as extensive: report the mean block group value both ways and compare it with the tract mean.

c. Compute the mean absolute error of the estimated population against the block group `pop` and report it.

In [16]:
# Task 6 starter

## Task 7 (10 points). Choose a method

For each situation, name the interpolation method you would use and justify it with the questions from lecture: Are the data at **points or polygons**? Is the variable a **count or a rate**? Do the source and target polygons **nest**? Is there enough spatial structure (how strong is the variogram, how many points)? Write 3 to 5 sentences per situation.

a. A city has 12 rain gauges and wants the rainfall at one house.

b. A researcher has 2010 tract counts of children under 5 and wants them on the 2020 tract boundaries.

Say what you would check before trusting each answer.

In [ ]:
# Task 7 starter

## Optional task (not graded). Write ordinary kriging yourself

Ordinary kriging for one target location with three stations is a 4 by 4 linear system. Build it with NumPy and solve it, then check your weights against `pyinterpolate`. Use the exponential model fitted in Task 4 (`expo`, with attributes `nugget`, `sill`, `rang`) and these three stations and target:

Use `target = xy[test][0]` and any three training points between 1 and 5 km from it as the stations `st`, with their temperatures `z`.

**Hints, in order.**

1. Distances: a 3 by 3 matrix `D` of station-to-station distances and a vector `d0` of station-to-target distances. `np.linalg.norm(a[:, None] - a[None], axis=2)` gives all pairwise distances.
2. Semivariance from distance: for the exponential model, $\gamma(h) = \text{nugget} + \text{sill}\,(1 - e^{-h/\text{range}})$ for $h > 0$ and $0$ for $h = 0$.
3. The left side: `G = np.ones((4, 4))`, put `gamma(D)` in the top-left 3 by 3 block, put the **nugget on the diagonal** of that block (this is how `pyinterpolate` handles it), and set `G[3, 3] = 0`.
4. The right side: `rhs = np.append(gamma(d0), 1.0)`.
5. Solve: `sol = np.linalg.solve(G, rhs)`. The first three entries are the weights $\lambda_i$ and the last is the Lagrange multiplier $\mu$.
6. Check: the three weights add to 1. The prediction is `lam @ z`, where `z` are the three station values.
7. The kriging variance is `lam @ gamma(d0) + mu`.
8. Compare with `pi.interpolate_points(expo, target[None], known_locations=np.column_stack([st, z]), no_neighbors=3, allow_approximate_solutions=True, progress_bar=False)`. The weights, prediction and variance should agree.

In [18]:
# Optional starter